# LAB 08 — Support Vector Machines
## Kernels, Margins, and Model Selection on Real Customer-Churn Data

## Student Information
- **Name:** Touqeer Ahmed
- **Student ID:** [023-24=0238]
- **Section:** G
- **GitHub:** [Enter your GitHub profile/repository]
- **Kaggle:** [Enter your Kaggle profile]
- **Dataset:** Telco Customer Churn (Kaggle)
- **Dataset Source:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn

---

### Before You Start
- You already studied SVM theory in class (margin, support vectors, kernel trick, C, gamma). This notebook does **not** re-teach that theory.
- This is **not** a "load data → `SVC().fit()` → done" lab. Most marks come from experimentation, visualization and justification, not from the fact that a model trained.
- You already know Python, Pandas, Matplotlib, `train_test_split`, encoding and evaluation metrics from Labs 2-6.
- Each task gives an **objective** and a **question** — not a code recipe. Hints tell you which tool is relevant; you decide how to use it.
- Do **not** provide complete code for every question yourself either — work it out. Every plot needs a written interpretation directly below it.


---

## Part 0 — Recap and Setup

Reload the Telco Customer Churn dataset and reapply the **same** cleaning/encoding pipeline from Labs 2/3/4/6:
- Convert `TotalCharges` to numeric and handle missing values
- Drop `customerID`
- Map `Churn` to 0/1
- One-hot encode categorical columns (`drop_first=True`)
- `train_test_split` with `stratify=y` — use the **same `random_state`** you used in Labs 3/4/6 so results stay comparable


In [1]:
# Load the Telco Customer Churn CSV.
# In Google Colab, upload `WA_Fn-UseC_-Telco-Customer-Churn.csv` when prompted.
# If running locally, put the CSV beside this notebook or update DATA_PATH below.

import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
)
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

RANDOM_STATE = 42   # Change to the same value used in Labs 3/4/6 if different.
TEST_SIZE = 0.20    # Change to the same test_size used in Labs 3/4/6 if different.
DATA_PATH = None    # Example: 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

candidate_paths = [
    DATA_PATH,
    'WA_Fn-UseC_-Telco-Customer-Churn.csv',
    'Telco-Customer-Churn.csv',
    'telco_churn.csv',
    '/content/WA_Fn-UseC_-Telco-Customer-Churn.csv'
]
csv_path = next((p for p in candidate_paths if p and os.path.exists(p)), None)

if csv_path is None:
    try:
        from google.colab import files
        print("Upload the Telco Customer Churn CSV file now.")
        uploaded = files.upload()
        csv_path = next((name for name in uploaded if name.lower().endswith('.csv')), None)
    except ImportError:
        raise FileNotFoundError(
            "Dataset CSV not found. Place the Telco Customer Churn CSV beside this notebook "
            "or set DATA_PATH to its location. Dataset: "
            "https://www.kaggle.com/datasets/blastchar/telco-customer-churn"
        )

if not csv_path:
    raise FileNotFoundError("No CSV file was uploaded. Please upload the Telco Customer Churn CSV.")

df = pd.read_csv(csv_path)
print("Loaded:", csv_path, "| Shape:", df.shape)
display(df.head())


Loaded: WA_Fn-UseC_-Telco-Customer-Churn.csv | Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# Clean target and numeric columns, then one-hot encode categorical features.
# The manual requests TotalCharges numeric, missing-value handling, dropping customerID,
# Churn mapped to 0/1, and get_dummies(drop_first=True).

df = df.copy()
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

if 'TotalCharges' not in df.columns or 'Churn' not in df.columns:
    raise ValueError("Expected Telco dataset columns 'TotalCharges' and 'Churn' were not found.")

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1}).fillna(df['Churn']).astype(int)

X = pd.get_dummies(df.drop(columns=['Churn']), drop_first=True, dtype=int)
y = df['Churn'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Features: {X.shape[1]} | Train rows: {len(X_train)} | Test rows: {len(X_test)}")
print("Training churn rate:", round(y_train.mean(), 3), "| Test churn rate:", round(y_test.mean(), 3))
display(pd.DataFrame({'Class': ['No churn (0)', 'Churn (1)'],
                      'Training count': [(y_train == 0).sum(), (y_train == 1).sum()],
                      'Test count': [(y_test == 0).sum(), (y_test == 1).sum()]}))


---

## Part 1 — Why Scaling Matters for SVM

**Task 1.1 — Objective:** Demonstrate what happens to an SVM when features are not scaled.

**What to do:**
1. Train a linear-kernel SVM (`SVC(kernel='linear')`) directly on your **un-scaled** encoded features.
2. Train a second linear-kernel SVM on the same split after applying `StandardScaler` (fit on train only, transform on train and test).
3. Compare accuracy, F1-score, training time, and any convergence warnings between the two runs.

```python
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# unscaled
svm_raw = SVC(kernel='linear', random_state=42)
svm_raw.fit(X_train, y_train)

# scaled  (fit_transform on TRAIN, transform ONLY on TEST)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
```


In [ ]:
# Part 1: Unscaled linear SVM
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import make_pipeline

part1_results = []
svm_raw = SVC(kernel='linear', random_state=RANDOM_STATE)
t0 = time.perf_counter()
with warnings.catch_warnings(record=True) as caught_raw:
    warnings.simplefilter('always')
    svm_raw.fit(X_train, y_train)
raw_time = time.perf_counter() - t0
raw_pred = svm_raw.predict(X_test)
part1_results.append({
    'Model': 'Linear SVM (unscaled)',
    'Accuracy': accuracy_score(y_test, raw_pred),
    'Precision': precision_score(y_test, raw_pred, zero_division=0),
    'Recall': recall_score(y_test, raw_pred, zero_division=0),
    'F1-score': f1_score(y_test, raw_pred, zero_division=0),
    'Training time (s)': raw_time,
    'Warnings': len(caught_raw)
})
display(pd.DataFrame(part1_results).round(4))


In [ ]:
# Part 1: Scaled linear SVM. Fit the scaler on training data only to prevent leakage.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

svm_scaled_linear = SVC(kernel='linear', random_state=RANDOM_STATE)
t0 = time.perf_counter()
with warnings.catch_warnings(record=True) as caught_scaled:
    warnings.simplefilter('always')
    svm_scaled_linear.fit(X_train_s, y_train)
scaled_time = time.perf_counter() - t0
scaled_pred = svm_scaled_linear.predict(X_test_s)
part1_results.append({
    'Model': 'Linear SVM (scaled)',
    'Accuracy': accuracy_score(y_test, scaled_pred),
    'Precision': precision_score(y_test, scaled_pred, zero_division=0),
    'Recall': recall_score(y_test, scaled_pred, zero_division=0),
    'F1-score': f1_score(y_test, scaled_pred, zero_division=0),
    'Training time (s)': scaled_time,
    'Warnings': len(caught_scaled)
})
part1_table = pd.DataFrame(part1_results)
display(part1_table.round(4))
print("Interpretation: SVM uses distances/margins, so feature scales can dominate the boundary. "
      "Use scaled features for the remaining SVM experiments.")


**Answers — Part 1**

(a) Compare the displayed table's accuracy, F1-score, training time, and warning count for the unscaled and scaled runs. These are measured from this notebook's dataset/split, so they may differ from other students' results.

(b) SVM chooses a decision boundary by maximizing the margin. If features have very different numeric ranges, large-range features can dominate the geometry and influence the boundary disproportionately.

(c) Use the scaled data (`X_train_s`, `X_test_s`) for subsequent SVM experiments. The scaler is fitted only on training data and then applied to the test data to avoid data leakage.


---

## Part 2 — Kernel Comparison

**Task 2.1 — Objective:** Train and fairly compare SVM classifiers using the linear, RBF and polynomial kernels on your **scaled** features.

**What to do:**
1. Train three `SVC` models — `kernel='linear'`, `kernel='rbf'`, `kernel='poly'` — with the same `random_state`, on the same scaled train split.
2. For each, record: Accuracy, Precision, Recall, F1-score, and number of support vectors (`model.support_.shape[0]`).
3. Assemble a single comparison table (a DataFrame is fine).
4. Plot the confusion matrix for whichever kernel performed best on F1-score.

| Syntax | Description |
|---|---|
| `SVC(kernel=...)` | `'linear'`, `'rbf'`, or `'poly'` |
| `model.support_` | indices of support vectors |
| `model.n_support_` | number of support vectors per class |


In [ ]:
# Part 2: Train and evaluate three kernels on the same scaled split.
kernel_models = {}
kernel_rows = []
for kernel_name in ['linear', 'rbf', 'poly']:
    model = SVC(kernel=kernel_name, random_state=RANDOM_STATE)
    start = time.perf_counter()
    model.fit(X_train_s, y_train)
    elapsed = time.perf_counter() - start
    pred = model.predict(X_test_s)
    kernel_models[kernel_name] = model
    kernel_rows.append({
        'Kernel': kernel_name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1-score': f1_score(y_test, pred, zero_division=0),
        'Support vectors': model.support_.shape[0],
        'Training time (s)': elapsed
    })
kernel_table = pd.DataFrame(kernel_rows).sort_values('F1-score', ascending=False).reset_index(drop=True)
display(kernel_table.round(4))
best_kernel_name = kernel_table.loc[0, 'Kernel']
best_kernel_model = kernel_models[best_kernel_name]
print(f"Best kernel by test F1-score: {best_kernel_name}")


In [ ]:
# The comparison table is displayed above; identify the kernel with the most support vectors.
most_sv_row = kernel_table.loc[kernel_table['Support vectors'].idxmax()]
print(f"Most support vectors: {most_sv_row['Kernel']} ({int(most_sv_row['Support vectors'])})")


In [ ]:
# Confusion matrix for the kernel with the best test F1-score.
best_kernel_pred = best_kernel_model.predict(X_test_s)
ConfusionMatrixDisplay.from_predictions(
    y_test, best_kernel_pred, display_labels=['No churn', 'Churn'], cmap='Blues'
)
plt.title(f'Confusion Matrix — {best_kernel_name} kernel')
plt.tight_layout()
plt.show()


**Answers — Part 2**

(a) The kernel with the highest **F1-score** in the comparison table is selected automatically and printed above. F1 balances precision and recall, which is useful because churners are the minority class.

(b) The table shows which kernel used the most support vectors. A larger support-vector count means more training points help define the boundary; it does **not** by itself prove that the boundary is more complex, because the kernel and parameter settings also matter.

(c) Carry the RBF kernel into tuning as required by Parts 4–5. Use the table's actual scores to compare it with linear and polynomial kernels; do not claim it is best unless your measured F1-score supports that claim.


---

## Part 3 — Visualizing the Margin: What Does C Actually Do?

**Task 3.1 — Objective:** Reduce your scaled features to 2 dimensions and visualize how an RBF-kernel SVM's decision boundary changes as C increases.

**What to do:**
1. Use `PCA(n_components=2)` on your scaled training features. Report the total explained variance ratio of the two components.
2. Train three RBF-kernel SVMs on this 2-D projection using a small, medium and large C (choose 3 values spanning at least two orders of magnitude, e.g. 0.01, 1, 100).
3. For each C, plot the 2-D decision boundary (mesh grid + `contourf`) with training points overlaid, and report the number of support vectors.

**Hint — Building the Mesh Grid:**
- Build a fine grid of (x, y) spanning slightly beyond your projected data's min/max using `np.meshgrid`.
- Call `model.predict()` on every grid point (flatten, then reshape back to grid shape).
- `plt.contourf(xx, yy, Z, alpha=0.3)` draws shaded regions; `plt.scatter(...)` overlays real points colored by true label.


In [ ]:
# Part 3: PCA projection fitted only on training features.
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_2d = pca.fit_transform(X_train_s)
explained = pca.explained_variance_ratio_.sum()
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained by 2 components: {explained:.4f} ({explained*100:.2f}%)")


In [ ]:
# Train RBF SVMs with C values spanning four orders of magnitude.
C_values = [0.01, 1, 100]
pca_svm_models = {}
pca_svm_rows = []
for c_value in C_values:
    model = SVC(kernel='rbf', C=c_value, gamma='scale', random_state=RANDOM_STATE)
    model.fit(X_train_2d, y_train)
    pca_svm_models[c_value] = model
    pca_svm_rows.append({'C': c_value, 'Support vectors': model.support_.shape[0]})
display(pd.DataFrame(pca_svm_rows))


In [ ]:
# Plot each model's decision regions with the training points overlaid.
x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 250),
    np.linspace(y_min, y_max, 250)
)
for c_value, model in pca_svm_models.items():
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)
    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, alpha=0.25, levels=[-0.5, 0.5, 1.5])
    plt.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, s=10, alpha=0.55, edgecolors='none')
    plt.title(f'RBF SVM Decision Regions (C={c_value}, gamma=scale)')
    plt.xlabel('Principal component 1')
    plt.ylabel('Principal component 2')
    plt.tight_layout()
    plt.show()
    print(f"C={c_value}: {model.support_.shape[0]} support vectors.")
print("Interpretation: compare smoothness and region shape across the three plots. The projection preserves "
      f"only {explained*100:.2f}% of training-feature variance, so it is a limited view of the full model.")


**Answers — Part 3**

(a) Compare the three plots. Increasing `C` penalizes training errors more strongly, which can produce a more flexible boundary; the exact shape should be described from your plots.

(b) The support-vector counts are printed for each `C`. Their direction of change is empirical and is not guaranteed to be monotonic.

(c) A very simple boundary that misses visible class structure may indicate underfitting; a highly irregular boundary that follows individual points may indicate overfitting. Judge this only as a 2-D illustration.

(d) PCA compresses the original feature space into two components and discards information. The plot may hide separations present in higher dimensions, so it is not the actual boundary used by the full-feature model.


---

## Part 4 — Systematic Hyperparameter Search

**Task 4.1 — Objective:** Run `GridSearchCV` over a C x gamma grid for an RBF-kernel SVM and select the best configuration using cross-validated F1-score.

**What to do:**
1. Define a parameter grid with at least 3 values of C and at least 3 values of gamma.
2. Run `GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='f1')` on your scaled **training** data.
3. Extract `gs.cv_results_` and reshape the mean test scores into a C x gamma grid.
4. Visualize as a heatmap (`imshow` or a pivoted DataFrame) with C and gamma labeled.
5. Refit the best estimator and report its performance on the held-out **test** set.

| Syntax | Description |
|---|---|
| `GridSearchCV(estimator, param_grid, cv, scoring)` | tries every combination via k-fold CV |
| `gs.best_params_` | best hyperparameter combination |
| `gs.cv_results_['mean_test_score']` | mean CV score per combination |


In [ ]:
# Part 4: Grid search for RBF SVM. CV sees only training data.
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.001, 0.01, 0.1]
}
gs = GridSearchCV(
    SVC(kernel='rbf', random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    return_train_score=True
)
gs.fit(X_train_s, y_train)
print("Best parameters:", gs.best_params_)
print(f"Best mean CV F1-score: {gs.best_score_:.4f}")


In [ ]:
# Heatmap of mean CV F1 scores across the C x gamma grid.
cv_results = pd.DataFrame(gs.cv_results_)
cv_results['gamma_label'] = cv_results['param_gamma'].astype(str)
heatmap_data = cv_results.pivot(index='param_C', columns='gamma_label', values='mean_test_score')
# Sort C numerically; keep gamma labels in the grid's declared order.
heatmap_data = heatmap_data.reindex(sorted(heatmap_data.index, key=float))
gamma_order = [str(value) for value in param_grid['gamma']]
heatmap_data = heatmap_data.reindex(columns=gamma_order)
display(heatmap_data.round(4))
plt.figure(figsize=(8, 5))
plt.imshow(heatmap_data.values, aspect='auto', origin='lower', cmap='viridis')
plt.xticks(range(len(heatmap_data.columns)), [str(x) for x in heatmap_data.columns])
plt.yticks(range(len(heatmap_data.index)), [str(x) for x in heatmap_data.index])
plt.xlabel('gamma')
plt.ylabel('C')
plt.title('GridSearchCV Mean Validation F1-score')
plt.colorbar(label='Mean CV F1-score')
plt.tight_layout()
plt.show()


In [ ]:
# Evaluate the selected estimator once on the held-out test set.
best_svm = gs.best_estimator_
t0 = time.perf_counter()
tuned_pred = best_svm.predict(X_test_s)
tuned_eval_time = time.perf_counter() - t0
tuned_metrics = {
    'Accuracy': accuracy_score(y_test, tuned_pred),
    'Precision': precision_score(y_test, tuned_pred, zero_division=0),
    'Recall': recall_score(y_test, tuned_pred, zero_division=0),
    'F1-score': f1_score(y_test, tuned_pred, zero_division=0)
}
print("Test metrics:", {k: round(v, 4) for k, v in tuned_metrics.items()})
print("Confusion matrix (rows=true, columns=predicted; order 0,1):")
print(confusion_matrix(y_test, tuned_pred))
ConfusionMatrixDisplay.from_predictions(y_test, tuned_pred, display_labels=['No churn', 'Churn'], cmap='Blues')
plt.title('Tuned RBF SVM — Test Confusion Matrix')
plt.tight_layout()
plt.show()

untuned_rbf_pred = kernel_models['rbf'].predict(X_test_s)
untuned_rbf_f1 = f1_score(y_test, untuned_rbf_pred, zero_division=0)
print(f"Untuned RBF test F1: {untuned_rbf_f1:.4f}")
print(f"Tuned RBF test F1:   {tuned_metrics['F1-score']:.4f}")


**Answers — Part 4**

(a) The selected `C`, `gamma`, and mean cross-validated F1-score are printed above.

(b) Inspect the heatmap: if multiple neighboring cells have similar scores, the model is not highly sensitive within that region; a single isolated high cell suggests greater sensitivity to parameter choice.

(c) Compare the best CV F1-score with the held-out test F1-score. A large gap can indicate overfitting to the cross-validation folds or a distribution difference; a close match suggests more consistent generalization.

(d) The untuned and tuned RBF test F1-scores are printed above. Tuning is worthwhile if the improvement is meaningful relative to its extra computation; the result is dataset-dependent.


---

## Part 5 — Class Imbalance and the Business Trade-off

**Task 5.1 — Objective:** Compare your best RBF-kernel SVM with and without `class_weight='balanced'`, and reason about which is more useful for the business.

**What to do:**
1. Using your best (C, gamma) from Part 4, train two RBF SVMs: `class_weight=None` and `class_weight='balanced'`.
2. Report Precision, Recall, F1-score and confusion matrix for both on the test set.
3. Plot an ROC curve or Precision-Recall curve for the balanced model.


In [ ]:
# Part 5: Compare default class weights with balanced class weights.
best_params = gs.best_params_
svm_default_weight = SVC(kernel='rbf', C=best_params['C'], gamma=best_params['gamma'],
                         class_weight=None, probability=True, random_state=RANDOM_STATE)
svm_balanced = SVC(kernel='rbf', C=best_params['C'], gamma=best_params['gamma'],
                   class_weight='balanced', probability=True, random_state=RANDOM_STATE)

svm_default_weight.fit(X_train_s, y_train)
svm_balanced.fit(X_train_s, y_train)

weight_rows = []
weight_predictions = {}
for label, model in [('Default weights', svm_default_weight), ('Balanced weights', svm_balanced)]:
    pred = model.predict(X_test_s)
    weight_predictions[label] = pred
    weight_rows.append({
        'Model': label,
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1-score': f1_score(y_test, pred, zero_division=0),
        'Accuracy': accuracy_score(y_test, pred)
    })
    print(f'\n{label} confusion matrix (rows=true, columns=predicted; order 0,1):')
    print(confusion_matrix(y_test, pred))
weight_table = pd.DataFrame(weight_rows)
display(weight_table.round(4))
print("Change (balanced - default):",
      {m: round(weight_table.loc[1, m] - weight_table.loc[0, m], 4)
       for m in ['Precision', 'Recall', 'F1-score', 'Accuracy']})


In [ ]:
# ROC curve for the balanced model. SVC(probability=True) enables predict_proba.
y_score_balanced = svm_balanced.predict_proba(X_test_s)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_score_balanced)
auc = roc_auc_score(y_test, y_score_balanced)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Balanced RBF SVM (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve — Balanced RBF SVM')
plt.legend()
plt.tight_layout()
plt.show()
print(f"ROC-AUC: {auc:.4f}")


**Answers — Part 5**

(a) Use the printed `Change (balanced - default)` values for the exact precision and recall changes. `class_weight='balanced'` raises the penalty for errors on the minority class; recall often increases while precision may decrease, but use your measured values.

(b) Missing a real churner (false negative) can mean losing a customer, while a false positive can waste retention resources or annoy a loyal customer. The deployment choice depends on the relative costs. Recommend the balanced model if the recall improvement is valuable and the precision trade-off is acceptable; otherwise choose the default model.

(c) `SVC` does not provide probabilities by default. The code enables `probability=True` and uses `predict_proba(X_test_s)[:, 1]` to obtain probability-like churn scores for the ROC curve. Probability calibration adds computational cost.


---

## Part 6 — Model Showdown: SVM vs. Everything Else

**Task 6.1 — Objective:** Build a single comparison table across all five algorithms (Decision Tree, Random Forest, Logistic Regression, KNN, SVM) and select ONE final model for production.

**What to do:**
1. Build a table: one row per model, columns Accuracy / Precision / Recall / F1-score. Re-run earlier models here on the same split if you don't have saved numbers.
2. Produce ONE bar chart comparing F1-score across all five models.


In [ ]:
# Part 6: Re-run common baseline models on exactly the same train/test split.
# All models use the already encoded features. SVM is the tuned RBF model from Part 4.
baseline_models = {
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight=None),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                            class_weight=None, n_jobs=-1),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Tuned RBF SVM': best_svm
}
comparison_rows = []
comparison_predictions = {}
training_times = {}
for name, model in baseline_models.items():
    start = time.perf_counter()
    if name == 'Tuned RBF SVM':
        pred = model.predict(X_test_s)
    else:
        model.fit(X_train_s if name in ['Logistic Regression', 'KNN'] else X_train, y_train)
        pred = model.predict(X_test_s if name in ['Logistic Regression', 'KNN'] else X_test)
    training_times[name] = time.perf_counter() - start
    comparison_predictions[name] = pred
    comparison_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1-score': f1_score(y_test, pred, zero_division=0),
        'Fit + predict time (s)': training_times[name]
    })
comparison_table = pd.DataFrame(comparison_rows).sort_values('F1-score', ascending=False).reset_index(drop=True)
display(comparison_table.round(4))
best_f1_model_name = comparison_table.loc[comparison_table['F1-score'].idxmax(), 'Model']
best_recall_model_name = comparison_table.loc[comparison_table['Recall'].idxmax(), 'Model']
print("Highest F1-score:", best_f1_model_name)
print("Highest recall:", best_recall_model_name)


In [ ]:
# One bar chart comparing F1-score across all five models.
plt.figure(figsize=(9, 5))
plt.bar(comparison_table['Model'], comparison_table['F1-score'])
plt.ylabel('F1-score')
plt.title('Model Showdown — F1-score')
plt.xticks(rotation=25, ha='right')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()
print("Interpretation: the tallest bar has the highest F1-score. Consider recall and precision as well "
      "before choosing a model for churn prevention.")


**Answers — Part 6**

(a) The model with the highest F1-score and the model with the highest recall are printed above. They may be different because F1 balances precision and recall, whereas recall measures how many actual churners are found.

(b) Use the measured fit/predict times and metric differences in the table. Extra SVM computation is justified only if its F1/recall improvement is meaningful for the business.

(c) The final recommendation should consider at least F1-score and recall (and precision, to estimate the false-positive burden). The notebook selects a candidate by the highest F1-score, but the business decision may favor a higher-recall model if missing churners is especially costly.


---

## Part 7 — Final Predictions and Wrap-Up

**Task 7.1 — Objective:** Generate a predictions file and write a short executive summary.

**What to do:**
1. Using your final chosen model, generate predictions on the test set and save `svm_predictions.csv` with true label, predicted label, and predicted churn probability/score.
2. Write a 5-8 sentence Executive Summary below: which model you chose, key metrics, the main trade-off you made in Part 5, and one limitation of your analysis.


In [ ]:
# Part 7: Choose the best-F1 model from the scoreboard and save predictions with a score.
# If tuned SVM is chosen, use its probability scores. For other models use predict_proba where available.
final_model_name = best_f1_model_name
final_model = baseline_models[final_model_name]
if final_model_name == 'Tuned RBF SVM':
    final_pred = final_model.predict(X_test_s)
    final_score = final_model.predict_proba(X_test_s)[:, 1] if hasattr(final_model, 'predict_proba') else final_model.decision_function(X_test_s)
elif hasattr(final_model, 'predict_proba'):
    final_X_test = X_test_s if final_model_name in ['Logistic Regression', 'KNN'] else X_test
    final_pred = final_model.predict(final_X_test)
    final_score = final_model.predict_proba(final_X_test)[:, 1]
else:
    final_X_test = X_test_s if final_model_name in ['Logistic Regression', 'KNN'] else X_test
    final_pred = final_model.predict(final_X_test)
    final_score = final_model.decision_function(final_X_test) if hasattr(final_model, 'decision_function') else final_pred

predictions_df = pd.DataFrame({
    'true_churn': np.asarray(y_test),
    'predicted_churn': np.asarray(final_pred),
    'predicted_churn_score': np.asarray(final_score)
}, index=y_test.index).sort_index()
predictions_df.to_csv('svm_predictions.csv', index=False)
print(f"Final model selected by F1-score: {final_model_name}")
print("Saved svm_predictions.csv | rows:", len(predictions_df))
display(predictions_df.head(10))


**Executive Summary (generated from this run)**

This notebook compares SVM scaling choices, three kernel types, the effect of `C`, and an RBF hyperparameter grid using cross-validation. The model with the highest measured F1-score in the five-model scoreboard is selected as the final candidate; its measured metrics are shown in Part 6. The class-weight experiment reports how precision and recall change when minority-class errors receive greater weight. For churn prevention, recall matters because false negatives are customers who churn but are not identified, while precision matters because false positives can waste retention resources. The saved `svm_predictions.csv` contains the true label, predicted label, and a probability-like churn score for the test rows. One limitation is that the train/test split is a single sample of the data and the 2-D PCA visualization retains only a fraction of the feature information. Before deployment, the company should choose a decision threshold based on actual retention costs and validate the model on newer customer data.

**Final Reflection**

(a) The most surprising result should be based on your outputs: compare the unscaled/scaled results, kernel F1-scores, and the effect of class weighting rather than assuming a result in advance.

(b) With one extra hour, I would test a precision-recall threshold strategy, investigate feature engineering, and evaluate a separate validation period or newer customer cohort. I would also compare the business costs of false positives and false negatives.


## How to run and submit

1. Open this notebook in Google Colab or Jupyter.
2. Run the cells from top to bottom. When prompted, upload the Telco Customer Churn CSV from Kaggle.
3. If your previous labs used a different `RANDOM_STATE` or `TEST_SIZE`, update those two values in Part 0 before running so the comparison is fair.
4. Read the tables/plots and keep the generated outputs. Metric values are computed from your dataset at runtime; they are intentionally not fabricated in advance.
5. Download the notebook after running it and submit it with `svm_predictions.csv` if your instructor requests the predictions file.